# Block 3 — Handwriting Recognition & Prior Fusion Engine
**Medical Document Intelligence System (Batch Pipeline)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RwaRwa599/epq3/blob/block3/block3/Block_3_Handwriting_Recognition.ipynb)

---
### Pipeline
1. **Ingest Block 2 ZIP:** Upload `block2_validated_batch.zip` (canonical sheets, crops, KG priors).
2. **Checkbox classification:** Density / slash / filled-box detector with HiTL flags.
3. **Handwriting recognition:** Digit/date parsers; optional TrOCR when `transformers` is installed.
4. **Prior fusion:** Re-rank drafts with Block 2 `assume()` candidates (profiles, tubes, write-ins).
5. **Export `block3_predictions_batch.zip`** for Block 4 / LIS.

## 1. Setup & Environment

In [ ]:
import os, sys, glob, json, zipfile, shutil

if 'google.colab' in sys.modules or os.path.exists('/content'):
    repo_dir = '/content/repo'
    if os.path.exists(repo_dir):
        !cd /content/repo && git fetch origin block3 && git reset --hard origin/block3
    else:
        !git clone -b block3 https://github.com/RwaRwa599/epq3.git /content/repo
    block3_path = '/content/repo/block3'
    if block3_path not in sys.path:
        sys.path.insert(0, block3_path)
    os.chdir(block3_path)

for mod in list(sys.modules.keys()):
    if mod.startswith('med_doc'):
        del sys.modules[mod]

!pip install -q "pydantic>=2.0.0" "numpy>=1.24.0" "opencv-python-headless>=4.8.0" "Pillow>=10.0.0" "matplotlib>=3.7.0"

from med_doc.htr import process_batch_from_block2, classify_mark
from med_doc.kg import KnowledgeGraph
print("✓ Environment initialized and med_doc.htr imported successfully!")

## 2. Upload Block 2 Batch ZIP (`block2_validated_batch.zip`)
The file popup asks for the ZIP produced by Block 2. If none is uploaded, a tiny mock batch is created so the rest of the notebook still runs.

In [ ]:
input_zip = "block2_validated_batch.zip"

try:
    from google.colab import files
    print("Upload 'block2_validated_batch.zip' from Block 2:")
    uploaded = files.upload()
    for filename in uploaded.keys():
        if filename.endswith(".zip"):
            input_zip = filename
            print(f"[+] Ingested ZIP: {input_zip}")
            break
except Exception:
    print("Interactive upload skipped (running locally).")

if not os.path.exists(input_zip):
    print("[!] No ZIP found. Building a mock Block 2 batch...")
    import numpy as np, cv2
    os.makedirs("mock_b2/docs/sample_sheet_01/crops/checkboxes", exist_ok=True)
    os.makedirs("mock_b2/docs/sample_sheet_01/crops/handwriting", exist_ok=True)
    canvas = np.full((200, 300, 3), 245, dtype=np.uint8)
    cv2.imwrite("mock_b2/docs/sample_sheet_01/canonical.png", canvas)
    empty = np.full((32, 32, 3), 245, dtype=np.uint8)
    empty[0:2,:] = 20; empty[-2:,:] = 20; empty[:,0:2] = 20; empty[:,-2:] = 20
    tick = empty.copy()
    for i in range(6, 26):
        tick[i, i] = 15
    cv2.imwrite("mock_b2/docs/sample_sheet_01/crops/checkboxes/cbc.png", tick)
    cv2.imwrite("mock_b2/docs/sample_sheet_01/crops/checkboxes/alt.png", empty)
    cv2.imwrite("mock_b2/docs/sample_sheet_01/crops/handwriting/others.png", empty)
    cv2.imwrite("mock_b2/docs/sample_sheet_01/crops/handwriting/tube_edta.png", empty)
    meta = {
        "doc_id": "sample_sheet_01",
        "fields": {
            "checkboxes": {
                "cbc": {"bbox": [10, 10, 42, 42], "crop_path": "crops/checkboxes/cbc.png"},
                "alt": {"bbox": [50, 10, 82, 42], "crop_path": "crops/checkboxes/alt.png"},
            },
            "handwriting": {
                "others": {"bbox": [10, 60, 120, 90], "crop_path": "crops/handwriting/others.png"},
                "tube_edta": {"bbox": [10, 100, 80, 130], "crop_path": "crops/handwriting/tube_edta.png"},
            },
        },
        "detected_marks": {
            "cbc": {"dark_ratio": 0.22, "is_marked_candidate": True},
            "alt": {"dark_ratio": 0.02, "is_marked_candidate": False},
        },
    }
    with open("mock_b2/docs/sample_sheet_01/metadata.json", "w") as f:
        json.dump(meta, f, indent=2)
    with open("mock_b2/docs/sample_sheet_01/prior_rankings.json", "w") as f:
        json.dump({"others": [], "tube_edta": [{"value": "1", "canonical_id": "tube_edta", "score": 0.9, "tier": 1, "reason": "prior"}]}, f)
    with open("mock_b2/docs/sample_sheet_01/validation_report.json", "w") as f:
        json.dump({"is_valid": True, "expected_tubes": {"EDTA": 1}, "implied_tests": [], "discrepancies": [], "warnings": []}, f)
    with open("mock_b2/manifest.json", "w") as f:
        json.dump({"version": "1.0", "block": "block2", "total_documents": 1, "documents": [{"doc_id": "sample_sheet_01"}]}, f)
    with zipfile.ZipFile(input_zip, "w") as zf:
        for root, _, files_list in os.walk("mock_b2"):
            for name in files_list:
                fp = os.path.join(root, name)
                zf.write(fp, os.path.relpath(fp, "mock_b2"))
    print(f"[+] Created demo {input_zip}")

print(f"\n✓ Ready to process: '{input_zip}' ({os.path.getsize(input_zip)/1024:.1f} KB)")

## 3. Run Batch HTR & Prior Fusion

In [ ]:
output_zip_path = "block3_predictions_batch.zip"
output_dir = "outputs/block3_batch"

result = process_batch_from_block2(
    input_source=input_zip,
    output_dir=output_dir,
    output_zip=output_zip_path,
    backend="lexicon",
)
manifest = result["manifest"]
print("=" * 70)
print(f"Total sheets: {manifest['total_documents']}")
print(f"Valid:        {manifest['valid_documents']}")
print(f"Need HiTL:    {manifest['hitl_documents']}")
print("=" * 70)

## 4. Inspect Predictions

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

for doc in manifest["documents"]:
    doc_id = doc["doc_id"]
    pred_path = f"{output_dir}/{doc['prediction_path']}"
    with open(pred_path) as f:
        pred = json.load(f)
    print(f"\n=== {doc_id} ===")
    print(f"  Valid: {pred['is_valid']}  confidence={pred['overall_confidence']}")
    print(f"  Ticked: {pred['ticked_test_ids']}")
    print(f"  HiTL fields: {pred['hitl_fields']}")
    print(f"  Expected tubes: {pred['expected_tubes']}")
    for fid, hw in list(pred['handwriting_fields'].items())[:6]:
        print(f"  HW {fid}: raw={hw['raw_text']!r} canonical={hw['canonical_value']!r} src={hw['source']}")
    canvas_path = f"{output_dir}/{doc['annotated_canvas_path']}"
    if os.path.exists(canvas_path):
        plt.figure(figsize=(10, 7))
        plt.imshow(Image.open(canvas_path))
        plt.title(f"Annotated canvas: {doc_id}")
        plt.axis("off")
        plt.show()

## 5. Download `block3_predictions_batch.zip` for Block 4

In [ ]:
try:
    from google.colab import files
    print(f"Downloading {output_zip_path} for Block 4...")
    files.download(output_zip_path)
except Exception:
    print(f"ZIP ready locally at: {os.path.abspath(output_zip_path)}")